# MedicalNLP-RAG — Phase 4D v2
## Task-Specific Confidence Estimation / Calibration

This notebook replaces the pooled Phase 4D confidence model.

The reason is methodological: BioASQ question types use different target metrics and naturally occupy very different score ranges. A pooled model can therefore obtain apparently strong discrimination mainly by learning the question type.

Phase 4D v2 removes this confounding by fitting a **separate confidence model for each BioASQ question type**.

The 860-question internal test remains locked.

---

## Frozen upstream pipeline

Nothing from Phases 3–4C is re-optimized:

`BM25 + MedCPT → RRF → MedCPT Cross-Encoder → Top-5 documents → Qwen3-4B-Instruct → Phase 4C v2 grounding`

Frozen evidence strategy:

`document_top5`

---

## Task-specific targets

| BioASQ type | Confidence target |
|---|---|
| yes/no | expected correctness (0/1) |
| factoid | expected MRR |
| list | expected F1 |
| summary | expected ROUGE-L |

For factoid, list, and summary, the confidence is an **expected benchmark task score**, not a probability that a medical statement is clinically true.

---

## Task-specific models

Four independent models are fitted:

- yes/no model: n = 232
- factoid model: n = 255
- list model: n = 170
- summary model: n = 205

Question type is **not used as a predictor**.

Each model uses the same inference-time feature family:

- question length;
- answer length;
- JSON validity;
- claim counts;
- Phase 4C grounding signals;
- MedCPT relevance signals;
- evidence-rank signals.

---

## Nested cross-fitting

For every task separately:

### Outer 10-fold
Produces held-out predictions for every validation question.

### Inner 5-fold
Within each outer training fold:

1. generate inner OOF Ridge predictions;
2. fit isotonic regression on those inner OOF predictions;
3. apply it to the untouched outer fold.

This prevents a question's own target from calibrating its reported confidence.

For yes/no, folds are stratified by the binary target.  
For other tasks, shuffled K-fold splitting is used.

---

## Calibration and ranking are kept distinct

The Ridge prediction is the **raw quality score**.

The isotonic transformation provides the **calibrated expected task score**.

For Phase 4E:

- ranking / tie-breaking can use the raw Ridge score;
- displayed confidence is the isotonic calibrated score.

Because isotonic regression is monotonic, it preserves ordering except where it creates plateaus.

---

## Baselines

Within each task we compare against:

- `train_mean_baseline`: outer-training-fold mean target;
- `mean_top1_entailment`;
- `top1_supported_claim_rate`;
- `task_ridge_raw`;
- `task_ridge_isotonic`.

No abstention threshold is selected in this notebook.

## 0. Configuration

In [ ]:
PROJECT_NAME = "MedicalNLP_RAG"
SEED = 42

FROZEN_EVIDENCE_STRATEGY = "document_top5"

TASKS = [
    "yesno",
    "factoid",
    "list",
    "summary",
]

OUTER_FOLDS = 10
INNER_FOLDS = 5

RIDGE_ALPHA = 1.0

CALIBRATION_BINS = 10
BOOTSTRAP_REPS = 5000

ALLOW_INTERNAL_TEST_EVALUATION = False

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2. Imports

In [ ]:
from pathlib import Path
import json
import random
import warnings
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import (
    KFold,
    StratifiedKFold,
)
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    log_loss,
)

from scipy.stats import spearmanr

warnings.filterwarnings("ignore")

random.seed(SEED)
np.random.seed(SEED)

## 3. Paths

In [ ]:
ROOT = Path(
    f"/content/drive/MyDrive/{PROJECT_NAME}"
)

PHASE4B_DIR = (
    ROOT / "results"
    / "phase4B_evidence_filtering"
)

PHASE4B_GEN_DIR = (
    PHASE4B_DIR / "generations"
)

PHASE4B_TABLE_DIR = (
    PHASE4B_DIR / "tables"
)

PHASE4C_DIR = (
    ROOT / "results"
    / "phase4C_grounding_v2"
)

PHASE4C_TABLE_DIR = (
    PHASE4C_DIR / "tables"
)

PHASE4D_V1_DIR = (
    ROOT / "results"
    / "phase4D_confidence_calibration"
)

PHASE4D_DIR = (
    ROOT / "results"
    / "phase4D_v2_task_specific_confidence"
)

TABLE_DIR = PHASE4D_DIR / "tables"
FIG_DIR = PHASE4D_DIR / "figures"
MODEL_DIR = PHASE4D_DIR / "models"
LOG_DIR = PHASE4D_DIR / "logs"

for p in [
    PHASE4D_DIR,
    TABLE_DIR,
    FIG_DIR,
    MODEL_DIR,
    LOG_DIR,
]:
    p.mkdir(
        parents=True,
        exist_ok=True
    )

PHASE4B_PER_QUESTION = (
    PHASE4B_TABLE_DIR
    / "phase4B_answer_scores_per_question.csv"
)

PHASE4B_GENERATIONS = (
    PHASE4B_GEN_DIR
    / "validation_862q_2filtered_conditions.jsonl"
)

QUESTION_GROUNDING = (
    PHASE4C_TABLE_DIR
    / "question_grounding_v2.csv"
)

CLAIM_GROUNDING = (
    PHASE4C_TABLE_DIR
    / "claim_grounding_v2.csv"
)

FROZEN_GROUNDING_METHOD = (
    PHASE4C_DIR
    / "frozen_grounding_method_v2.json"
)

POOLED_V1_SUMMARY = (
    PHASE4D_V1_DIR
    / "tables"
    / "confidence_model_summary.csv"
)

for required in [
    PHASE4B_PER_QUESTION,
    PHASE4B_GENERATIONS,
    QUESTION_GROUNDING,
    CLAIM_GROUNDING,
    FROZEN_GROUNDING_METHOD,
]:
    assert required.exists(), required

print(
    "Phase 4D v2 output:",
    PHASE4D_DIR
)

## 4. Locked-test safety

In [ ]:
if ALLOW_INTERNAL_TEST_EVALUATION:
    raise RuntimeError(
        "Internal test must remain locked during Phase 4D v2."
    )

print("INTERNAL TEST REMAINS LOCKED.")

## 5. Verify frozen Phase 4C method

In [ ]:
with open(
    FROZEN_GROUNDING_METHOD,
    "r",
    encoding="utf-8"
) as f:
    grounding_method = json.load(f)

assert (
    grounding_method[
        "frozen_evidence_strategy"
    ]
    == FROZEN_EVIDENCE_STRATEGY
)

assert (
    grounding_method[
        "internal_test_evaluated"
    ]
    is False
)

print(
    "Frozen evidence strategy verified:",
    FROZEN_EVIDENCE_STRATEGY
)

## 6. Read JSONL helper

In [ ]:
def read_jsonl(path):
    rows = []

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:
        for line in f:
            if line.strip():
                rows.append(
                    json.loads(line)
                )

    return pd.DataFrame(rows)

## 7. Load frozen Top-5 generations

In [ ]:
generation_df = read_jsonl(
    PHASE4B_GENERATIONS
)

generation_df["id"] = (
    generation_df["id"]
    .astype(str)
)

generation_df = (
    generation_df[
        generation_df[
            "condition"
        ]
        == FROZEN_EVIDENCE_STRATEGY
    ]
    .copy()
)

assert len(generation_df) == 862

display(
    generation_df[
        "type"
    ].value_counts()
)

## 8. Load Phase 4B task scores

In [ ]:
score_df = pd.read_csv(
    PHASE4B_PER_QUESTION,
    dtype={"id": str}
)

score_df = (
    score_df[
        score_df[
            "condition"
        ]
        == FROZEN_EVIDENCE_STRATEGY
    ]
    .copy()
)

assert len(score_df) == 862

print(
    "Available score columns:",
    score_df.columns.tolist()
)

## 9. Define task-specific target

In [ ]:
def find_first_column(
    frame,
    candidates
):
    for col in candidates:
        if col in frame.columns:
            return col

    raise KeyError(
        "None of these columns were found: "
        + ", ".join(candidates)
    )


YESNO_COL = find_first_column(
    score_df,
    [
        "yesno_correct",
        "yesno_accuracy",
    ]
)

FACTOID_COL = find_first_column(
    score_df,
    [
        "factoid_mrr",
    ]
)

LIST_COL = find_first_column(
    score_df,
    [
        "list_f1",
    ]
)

SUMMARY_COL = find_first_column(
    score_df,
    [
        "rougeL_f",
        "summary_rougeL_f",
    ]
)


def get_task_score(row):
    qtype = str(
        row["type"]
    ).lower()

    if qtype == "yesno":
        return float(
            row[YESNO_COL]
        )

    if qtype == "factoid":
        return float(
            row[FACTOID_COL]
        )

    if qtype == "list":
        return float(
            row[LIST_COL]
        )

    if qtype == "summary":
        return float(
            row[SUMMARY_COL]
        )

    raise ValueError(qtype)


score_df[
    "task_score"
] = score_df.apply(
    get_task_score,
    axis=1
)

assert score_df[
    "task_score"
].between(
    0,
    1
).all()

display(
    score_df.groupby(
        "type"
    )["task_score"]
    .describe()
    .round(4)
)

## 10. Load Phase 4C grounding tables

In [ ]:
question_grounding_df = pd.read_csv(
    QUESTION_GROUNDING,
    dtype={"id": str}
)

question_grounding_df = (
    question_grounding_df[
        question_grounding_df[
            "condition"
        ]
        == FROZEN_EVIDENCE_STRATEGY
    ]
    .copy()
)

claim_grounding_df = pd.read_csv(
    CLAIM_GROUNDING,
    dtype={
        "id": str,
        "top1_pmid": str,
    }
)

claim_grounding_df = (
    claim_grounding_df[
        claim_grounding_df[
            "condition"
        ]
        == FROZEN_EVIDENCE_STRATEGY
    ]
    .copy()
)

assert len(
    question_grounding_df
) == 862

print(
    "Frozen Top-5 claims:",
    len(claim_grounding_df)
)

# Part A — Build inference-time features

No gold-answer score is used as an input.

Question type is **not** included as a feature.

## 11. Generated-answer length features

In [ ]:
def word_count(value):
    text = str(
        value
        if value is not None
        else ""
    ).strip()

    return len(
        text.split()
    )


required_generation_cols = [
    "id",
    "type",
    "ideal_answer_pred",
    "parse_valid",
]

missing = [
    col
    for col in required_generation_cols
    if col not in generation_df.columns
]

assert not missing, (
    f"Missing generation columns: {missing}"
)

generation_features = (
    generation_df[
        required_generation_cols
    ]
    .copy()
)

generation_features["id"] = (
    generation_features["id"]
    .astype(str)
)

# Recover question text from generation or score table.
question_source = None
question_col = None

for candidate in [
    "body",
    "question",
]:
    if candidate in generation_df.columns:
        question_source = generation_df
        question_col = candidate
        break

if question_source is None:
    for candidate in [
        "question",
        "body",
    ]:
        if candidate in score_df.columns:
            question_source = score_df
            question_col = candidate
            break

if question_source is None:
    raise RuntimeError(
        "Question text could not be found."
    )

question_text_df = (
    question_source[
        [
            "id",
            "type",
            question_col,
        ]
    ]
    .copy()
    .rename(
        columns={
            question_col:
            "question_text"
        }
    )
)

question_text_df["id"] = (
    question_text_df["id"]
    .astype(str)
)

question_text_df = (
    question_text_df
    .drop_duplicates(
        [
            "id",
            "type",
        ]
    )
)

generation_features = (
    generation_features.merge(
        question_text_df,
        on=[
            "id",
            "type",
        ],
        how="left",
        validate="one_to_one",
    )
)

assert len(
    generation_features
) == 862

assert (
    generation_features[
        "question_text"
    ]
    .notna()
    .all()
)

generation_features[
    "question_words"
] = (
    generation_features[
        "question_text"
    ]
    .map(word_count)
)

generation_features[
    "ideal_answer_words"
] = (
    generation_features[
        "ideal_answer_pred"
    ]
    .map(word_count)
)

generation_features[
    "parse_valid"
] = (
    generation_features[
        "parse_valid"
    ]
    .astype(int)
)

generation_features = (
    generation_features.drop(
        columns=[
            "question_text",
            "ideal_answer_pred",
        ]
    )
)

print(
    "Question text source:",
    question_col
)

print(
    "Rows:",
    len(generation_features)
)

display(
    generation_features.head()
)

## 12. Claim-level relevance aggregates

In [ ]:
claim_feature_df = (
    claim_grounding_df.groupby(
        [
            "id",
            "type",
        ]
    )
    .agg(
        claim_count=(
            "claim_id",
            "size"
        ),
        mean_top1_medcpt_score=(
            "top1_medcpt_score",
            "mean"
        ),
        min_top1_medcpt_score=(
            "top1_medcpt_score",
            "min"
        ),
        mean_top1_document_rank=(
            "top1_document_rank",
            "mean"
        ),
        rank1_document_fraction=(
            "top1_document_rank",
            lambda x: float(
                np.mean(
                    np.asarray(x)
                    == 1
                )
            )
        ),
        min_top1_entailment_claim=(
            "top1_entailment",
            "min"
        ),
        max_top1_contradiction_claim=(
            "top1_contradiction",
            "max"
        ),
    )
    .reset_index()
)

display(
    claim_feature_df.head()
)

## 13. Merge full task-specific modeling table

In [ ]:
grounding_columns = [
    "id",
    "type",
    "n_claims",
    "mean_top1_entailment",
    "mean_top1_contradiction",
    "top1_supported_claim_rate",
    "top1_contradiction_rate",
    "mean_top3_max_entailment",
    "top3_supported_claim_rate",
    "fully_top1_supported",
    "fully_top3_supported",
]

missing_grounding = [
    col
    for col in grounding_columns
    if col not in
    question_grounding_df.columns
]

assert not missing_grounding, (
    missing_grounding
)

model_df = (
    generation_features
    .merge(
        question_grounding_df[
            grounding_columns
        ],
        on=[
            "id",
            "type",
        ],
        validate="one_to_one",
    )
    .merge(
        claim_feature_df,
        on=[
            "id",
            "type",
        ],
        how="left",
        validate="one_to_one",
    )
    .merge(
        score_df[
            [
                "id",
                "type",
                "task_score",
            ]
        ],
        on=[
            "id",
            "type",
        ],
        validate="one_to_one",
    )
)

assert len(model_df) == 862

numeric_fill_zero = [
    "claim_count",
    "mean_top1_medcpt_score",
    "min_top1_medcpt_score",
    "mean_top1_document_rank",
    "rank1_document_fraction",
    "min_top1_entailment_claim",
    "max_top1_contradiction_claim",
]

for col in numeric_fill_zero:
    model_df[col] = (
        model_df[col]
        .fillna(0.0)
    )

model_df.to_csv(
    TABLE_DIR
    / "phase4D_v2_modeling_table.csv",
    index=False
)

display(
    model_df.groupby(
        "type"
    )["task_score"]
    .agg(
        [
            "count",
            "mean",
            "std",
        ]
    )
    .round(4)
)

## 14. Feature specification — question type excluded

In [ ]:
NUMERIC_FEATURES = [
    "question_words",
    "ideal_answer_words",
    "parse_valid",

    "n_claims",
    "mean_top1_entailment",
    "mean_top1_contradiction",
    "top1_supported_claim_rate",
    "top1_contradiction_rate",
    "mean_top3_max_entailment",
    "top3_supported_claim_rate",
    "fully_top1_supported",
    "fully_top3_supported",

    "mean_top1_medcpt_score",
    "min_top1_medcpt_score",
    "mean_top1_document_rank",
    "rank1_document_fraction",
    "min_top1_entailment_claim",
    "max_top1_contradiction_claim",
]

assert "type" not in NUMERIC_FEATURES

print(
    "Task-specific numeric features:",
    NUMERIC_FEATURES
)

## 15. Save environment

In [ ]:
environment = {
    "timestamp_utc": (
        datetime.now(
            timezone.utc
        ).isoformat()
    ),
    "dataset": "BioASQ14b",
    "validation_questions": 862,
    "internal_test_evaluated": False,
    "frozen_evidence_strategy": (
        FROZEN_EVIDENCE_STRATEGY
    ),
    "design": (
        "four independent task-specific "
        "confidence models"
    ),
    "question_type_used_as_feature": False,
    "target_mapping": {
        "yesno": YESNO_COL,
        "factoid": FACTOID_COL,
        "list": LIST_COL,
        "summary": SUMMARY_COL,
    },
    "outer_folds": OUTER_FOLDS,
    "inner_folds": INNER_FOLDS,
    "ridge_alpha": RIDGE_ALPHA,
    "calibration": (
        "nested cross-fitted isotonic regression"
    ),
    "ranking_signal": "task_ridge_raw",
    "calibrated_confidence": (
        "task_ridge_isotonic"
    ),
    "features": NUMERIC_FEATURES,
}

with open(
    LOG_DIR / "environment.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        environment,
        f,
        indent=2,
        ensure_ascii=False
    )

print(
    json.dumps(
        environment,
        indent=2,
        ensure_ascii=False
    )
)

# Part B — Task-specific nested cross-fitting

A separate Ridge + isotonic pipeline is estimated inside each question type.

## 16. Ridge pipeline

In [ ]:
def make_ridge_pipeline():
    preprocessor = (
        ColumnTransformer(
            transformers=[
                (
                    "numeric",
                    StandardScaler(),
                    NUMERIC_FEATURES,
                ),
            ],
            remainder="drop",
        )
    )

    return Pipeline(
        steps=[
            (
                "preprocessor",
                preprocessor,
            ),
            (
                "ridge",
                Ridge(
                    alpha=RIDGE_ALPHA
                ),
            ),
        ]
    )

## 17. Task-specific split helpers

In [ ]:
def make_outer_splitter(
    task,
    seed,
):
    if task == "yesno":
        return StratifiedKFold(
            n_splits=OUTER_FOLDS,
            shuffle=True,
            random_state=seed,
        )

    return KFold(
        n_splits=OUTER_FOLDS,
        shuffle=True,
        random_state=seed,
    )


def make_inner_splitter(
    task,
    seed,
):
    if task == "yesno":
        return StratifiedKFold(
            n_splits=INNER_FOLDS,
            shuffle=True,
            random_state=seed,
        )

    return KFold(
        n_splits=INNER_FOLDS,
        shuffle=True,
        random_state=seed,
    )


def split_iterator(
    splitter,
    X,
    y,
    task,
):
    if task == "yesno":
        return splitter.split(
            X,
            y.astype(int),
        )

    return splitter.split(
        X
    )

## 18. Nested task-specific cross-fitting

In [ ]:
prediction_parts = []
fold_audit_rows = []

for task_index, task in enumerate(
    TASKS
):
    task_df = (
        model_df[
            model_df[
                "type"
            ] == task
        ]
        .copy()
        .reset_index(drop=True)
    )

    X_task = task_df[
        NUMERIC_FEATURES
    ].copy()

    y_task = task_df[
        "task_score"
    ].to_numpy(float)

    n_task = len(
        task_df
    )

    oof_raw = np.full(
        n_task,
        np.nan,
        dtype=float
    )

    oof_iso = np.full(
        n_task,
        np.nan,
        dtype=float
    )

    oof_train_mean = np.full(
        n_task,
        np.nan,
        dtype=float
    )

    outer_fold = np.full(
        n_task,
        -1,
        dtype=int
    )

    outer_cv = make_outer_splitter(
        task,
        SEED + 100 * task_index,
    )

    outer_splits = split_iterator(
        outer_cv,
        X_task,
        y_task,
        task,
    )

    for outer_fold_id, (
        train_idx,
        test_idx
    ) in enumerate(
        outer_splits
    ):
        X_train = (
            X_task.iloc[
                train_idx
            ]
            .reset_index(drop=True)
        )

        y_train = (
            y_task[
                train_idx
            ]
        )

        X_test = (
            X_task.iloc[
                test_idx
            ]
        )

        # Baseline predicted from training fold only.
        train_mean = float(
            np.mean(
                y_train
            )
        )

        oof_train_mean[
            test_idx
        ] = train_mean

        # Inner OOF raw predictions for calibration.
        inner_oof_raw = np.full(
            len(train_idx),
            np.nan,
            dtype=float
        )

        inner_cv = make_inner_splitter(
            task,
            (
                SEED
                + 1000
                + 100 * task_index
                + outer_fold_id
            ),
        )

        inner_splits = split_iterator(
            inner_cv,
            X_train,
            y_train,
            task,
        )

        for (
            inner_train_idx,
            inner_val_idx
        ) in inner_splits:
            inner_model = (
                make_ridge_pipeline()
            )

            inner_model.fit(
                X_train.iloc[
                    inner_train_idx
                ],
                y_train[
                    inner_train_idx
                ],
            )

            inner_oof_raw[
                inner_val_idx
            ] = (
                inner_model.predict(
                    X_train.iloc[
                        inner_val_idx
                    ]
                )
            )

        assert np.isfinite(
            inner_oof_raw
        ).all()

        # Fit monotonic calibration map only on inner OOF.
        isotonic = (
            IsotonicRegression(
                y_min=0.0,
                y_max=1.0,
                out_of_bounds="clip",
            )
        )

        isotonic.fit(
            inner_oof_raw,
            y_train,
        )

        outer_model = (
            make_ridge_pipeline()
        )

        outer_model.fit(
            X_train,
            y_train,
        )

        raw_test = (
            outer_model.predict(
                X_test
            )
        )

        iso_test = (
            isotonic.transform(
                raw_test
            )
        )

        oof_raw[
            test_idx
        ] = np.clip(
            raw_test,
            0.0,
            1.0
        )

        oof_iso[
            test_idx
        ] = np.clip(
            iso_test,
            0.0,
            1.0
        )

        outer_fold[
            test_idx
        ] = outer_fold_id

        fold_audit_rows.append({
            "task": task,
            "outer_fold": int(
                outer_fold_id
            ),
            "train_n": int(
                len(train_idx)
            ),
            "test_n": int(
                len(test_idx)
            ),
            "train_target_mean": (
                train_mean
            ),
            "test_target_mean": float(
                np.mean(
                    y_task[
                        test_idx
                    ]
                )
            ),
        })

        print(
            f"{task:8s} "
            f"outer_fold={outer_fold_id} "
            f"train={len(train_idx)} "
            f"test={len(test_idx)}"
        )

    assert np.isfinite(
        oof_raw
    ).all()

    assert np.isfinite(
        oof_iso
    ).all()

    assert np.isfinite(
        oof_train_mean
    ).all()

    assert (
        outer_fold >= 0
    ).all()

    task_pred = task_df[
        [
            "id",
            "type",
            "task_score",
            "mean_top1_entailment",
            "top1_supported_claim_rate",
        ]
    ].copy()

    task_pred[
        "outer_fold"
    ] = outer_fold

    task_pred[
        "train_mean_baseline"
    ] = oof_train_mean

    task_pred[
        "task_ridge_raw"
    ] = oof_raw

    task_pred[
        "task_ridge_isotonic"
    ] = oof_iso

    prediction_parts.append(
        task_pred
    )

task_oof_df = pd.concat(
    prediction_parts,
    ignore_index=True
)

assert len(
    task_oof_df
) == 862

fold_audit_df = pd.DataFrame(
    fold_audit_rows
)

task_oof_df.to_csv(
    TABLE_DIR
    / "task_specific_oof_predictions.csv",
    index=False
)

fold_audit_df.to_csv(
    TABLE_DIR
    / "task_specific_fold_audit.csv",
    index=False
)

print(
    "Task-specific nested cross-fitting complete."
)

# Part C — Calibration metrics

## 19. Reliability table helper

In [ ]:
def calibration_table(
    y_true,
    y_pred,
    n_bins=CALIBRATION_BINS,
):
    y_true = np.asarray(
        y_true,
        dtype=float
    )

    y_pred = np.clip(
        np.asarray(
            y_pred,
            dtype=float
        ),
        0.0,
        1.0,
    )

    edges = np.linspace(
        0.0,
        1.0,
        n_bins + 1
    )

    bin_id = np.digitize(
        y_pred,
        edges[1:-1],
        right=False
    )

    rows = []

    for b in range(
        n_bins
    ):
        mask = (
            bin_id == b
        )

        count = int(
            mask.sum()
        )

        if count == 0:
            continue

        mean_pred = float(
            np.mean(
                y_pred[
                    mask
                ]
            )
        )

        mean_obs = float(
            np.mean(
                y_true[
                    mask
                ]
            )
        )

        rows.append({
            "bin": int(b),
            "bin_low": float(
                edges[b]
            ),
            "bin_high": float(
                edges[b + 1]
            ),
            "n": count,
            "mean_confidence": (
                mean_pred
            ),
            "mean_observed_task_score": (
                mean_obs
            ),
            "gap": float(
                mean_pred
                - mean_obs
            ),
            "absolute_gap": float(
                abs(
                    mean_pred
                    - mean_obs
                )
            ),
            "weight": float(
                count
                / len(y_true)
            ),
        })

    return pd.DataFrame(
        rows
    )

## 20. Metric helper

In [ ]:
def confidence_metrics(
    y_true,
    y_pred,
):
    y_true = np.asarray(
        y_true,
        dtype=float
    )

    y_pred = np.clip(
        np.asarray(
            y_pred,
            dtype=float
        ),
        0.0,
        1.0,
    )

    cal = calibration_table(
        y_true,
        y_pred,
    )

    ece = float(
        np.sum(
            cal[
                "weight"
            ]
            * cal[
                "absolute_gap"
            ]
        )
    )

    mae = float(
        mean_absolute_error(
            y_true,
            y_pred
        )
    )

    rmse = float(
        np.sqrt(
            mean_squared_error(
                y_true,
                y_pred
            )
        )
    )

    bias = float(
        np.mean(y_pred)
        - np.mean(y_true)
    )

    if np.std(
        y_pred
    ) > 1e-12:
        slope, intercept = np.polyfit(
            y_pred,
            y_true,
            1
        )
    else:
        slope = np.nan
        intercept = np.nan

    if (
        np.std(y_pred) > 1e-12
        and np.std(y_true) > 1e-12
    ):
        rho, rho_p = spearmanr(
            y_pred,
            y_true
        )
    else:
        rho = np.nan
        rho_p = np.nan

    return {
        "mae": mae,
        "rmse": rmse,
        "ece_10bin": ece,
        "calibration_bias": bias,
        "calibration_intercept": float(
            intercept
        ),
        "calibration_slope": float(
            slope
        ),
        "spearman_rho": float(
            rho
        ),
        "spearman_p": float(
            rho_p
        ),
        "mean_confidence": float(
            np.mean(
                y_pred
            )
        ),
        "mean_observed_task_score": float(
            np.mean(
                y_true
            )
        ),
    }

## 21. Evaluate every method within each task

In [ ]:
METHODS = [
    "train_mean_baseline",
    "mean_top1_entailment",
    "top1_supported_claim_rate",
    "task_ridge_raw",
    "task_ridge_isotonic",
]

metric_rows = []

for task in TASKS:
    subset = (
        task_oof_df[
            task_oof_df[
                "type"
            ] == task
        ]
    )

    for method in METHODS:
        metrics = (
            confidence_metrics(
                subset[
                    "task_score"
                ],
                subset[
                    method
                ],
            )
        )

        metrics.update({
            "task": task,
            "method": method,
            "n": int(
                len(subset)
            ),
        })

        metric_rows.append(
            metrics
        )

task_metric_df = pd.DataFrame(
    metric_rows
)

task_metric_df.to_csv(
    TABLE_DIR
    / "task_specific_confidence_metrics.csv",
    index=False
)

display(
    task_metric_df[
        [
            "task",
            "method",
            "n",
            "mae",
            "rmse",
            "ece_10bin",
            "calibration_bias",
            "calibration_slope",
            "spearman_rho",
            "spearman_p",
        ]
    ]
)

## 22. Task-specific Ridge raw vs isotonic summary

In [ ]:
ridge_comparison = (
    task_metric_df[
        task_metric_df[
            "method"
        ].isin([
            "task_ridge_raw",
            "task_ridge_isotonic",
        ])
    ]
    .copy()
)

display(
    ridge_comparison[
        [
            "task",
            "method",
            "mae",
            "rmse",
            "ece_10bin",
            "calibration_bias",
            "calibration_slope",
            "spearman_rho",
        ]
    ]
)

## 23. Yes/no binary calibration

In [ ]:
yesno_df = (
    task_oof_df[
        task_oof_df[
            "type"
        ] == "yesno"
    ]
    .copy()
)

assert set(
    np.unique(
        yesno_df[
            "task_score"
        ]
    )
).issubset({
    0.0,
    1.0,
})

yesno_rows = []

for method in METHODS:
    y_true = (
        yesno_df[
            "task_score"
        ]
        .to_numpy(float)
    )

    y_prob = np.clip(
        yesno_df[
            method
        ].to_numpy(float),
        1e-6,
        1 - 1e-6,
    )

    yesno_rows.append({
        "method": method,
        "n": int(
            len(yesno_df)
        ),
        "brier_score": float(
            np.mean(
                (
                    y_prob
                    - y_true
                ) ** 2
            )
        ),
        "log_loss": float(
            log_loss(
                y_true,
                y_prob,
                labels=[
                    0,
                    1,
                ]
            )
        ),
        "ece_10bin": float(
            confidence_metrics(
                y_true,
                y_prob
            )[
                "ece_10bin"
            ]
        ),
    })

yesno_binary_df = (
    pd.DataFrame(
        yesno_rows
    )
    .sort_values(
        "brier_score"
    )
)

yesno_binary_df.to_csv(
    TABLE_DIR
    / "yesno_task_specific_binary_calibration.csv",
    index=False
)

display(
    yesno_binary_df
)

# Part D — Macro task-level summary

Macro averaging gives each BioASQ task equal weight and prevents the larger groups from dominating the summary.

Spearman is macro-averaged only across finite task-level values.

## 24. Macro-average task-specific metrics

In [ ]:
macro_rows = []

for method in METHODS:
    subset = (
        task_metric_df[
            task_metric_df[
                "method"
            ] == method
        ]
    )

    macro_rows.append({
        "method": method,
        "macro_mae": float(
            subset[
                "mae"
            ].mean()
        ),
        "macro_rmse": float(
            subset[
                "rmse"
            ].mean()
        ),
        "macro_ece_10bin": float(
            subset[
                "ece_10bin"
            ].mean()
        ),
        "macro_abs_calibration_bias": float(
            subset[
                "calibration_bias"
            ].abs().mean()
        ),
        "macro_spearman_rho": float(
            subset[
                "spearman_rho"
            ].dropna().mean()
        ),
    })

macro_summary = (
    pd.DataFrame(
        macro_rows
    )
    .sort_values(
        [
            "macro_rmse",
            "macro_ece_10bin",
        ]
    )
    .reset_index(drop=True)
)

macro_summary.to_csv(
    TABLE_DIR
    / "task_macro_confidence_summary.csv",
    index=False
)

display(
    macro_summary
)

# Part E — Reliability diagrams by task

## 25. Save reliability-bin tables

In [ ]:
reliability_parts = []

for task in TASKS:
    subset = (
        task_oof_df[
            task_oof_df[
                "type"
            ] == task
        ]
    )

    for method in [
        "task_ridge_raw",
        "task_ridge_isotonic",
    ]:
        cal = (
            calibration_table(
                subset[
                    "task_score"
                ],
                subset[
                    method
                ],
            )
        )

        cal[
            "task"
        ] = task

        cal[
            "method"
        ] = method

        reliability_parts.append(
            cal
        )

reliability_df = pd.concat(
    reliability_parts,
    ignore_index=True
)

reliability_df.to_csv(
    TABLE_DIR
    / "task_specific_reliability_bins.csv",
    index=False
)

display(
    reliability_df.head()
)

## 26. Reliability figure — yes/no

In [ ]:
task = "yesno"

subset = (
    task_oof_df[
        task_oof_df[
            "type"
        ] == task
    ]
)

fig, ax = plt.subplots(
    figsize=(6.4, 6.0)
)

ax.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Perfect calibration",
)

for method in [
    "task_ridge_raw",
    "task_ridge_isotonic",
]:
    cal = calibration_table(
        subset[
            "task_score"
        ],
        subset[
            method
        ],
    )

    ax.plot(
        cal[
            "mean_confidence"
        ],
        cal[
            "mean_observed_task_score"
        ],
        marker="o",
        label=method,
    )

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

ax.set_xlabel(
    "Predicted confidence"
)

ax.set_ylabel(
    "Observed correctness rate"
)

ax.set_title(
    "Yes/no task-specific reliability"
)

ax.legend()

fig.tight_layout()

fig.savefig(
    FIG_DIR
    / "fig01_yesno_reliability.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## 27. Reliability figure — factoid

In [ ]:
task = "factoid"

subset = (
    task_oof_df[
        task_oof_df[
            "type"
        ] == task
    ]
)

fig, ax = plt.subplots(
    figsize=(6.4, 6.0)
)

ax.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Perfect calibration",
)

for method in [
    "task_ridge_raw",
    "task_ridge_isotonic",
]:
    cal = calibration_table(
        subset[
            "task_score"
        ],
        subset[
            method
        ],
    )

    ax.plot(
        cal[
            "mean_confidence"
        ],
        cal[
            "mean_observed_task_score"
        ],
        marker="o",
        label=method,
    )

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

ax.set_xlabel(
    "Predicted expected MRR"
)

ax.set_ylabel(
    "Observed mean MRR"
)

ax.set_title(
    "Factoid task-specific reliability"
)

ax.legend()

fig.tight_layout()

fig.savefig(
    FIG_DIR
    / "fig02_factoid_reliability.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## 28. Reliability figure — list

In [ ]:
task = "list"

subset = (
    task_oof_df[
        task_oof_df[
            "type"
        ] == task
    ]
)

fig, ax = plt.subplots(
    figsize=(6.4, 6.0)
)

ax.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Perfect calibration",
)

for method in [
    "task_ridge_raw",
    "task_ridge_isotonic",
]:
    cal = calibration_table(
        subset[
            "task_score"
        ],
        subset[
            method
        ],
    )

    ax.plot(
        cal[
            "mean_confidence"
        ],
        cal[
            "mean_observed_task_score"
        ],
        marker="o",
        label=method,
    )

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

ax.set_xlabel(
    "Predicted expected F1"
)

ax.set_ylabel(
    "Observed mean F1"
)

ax.set_title(
    "List task-specific reliability"
)

ax.legend()

fig.tight_layout()

fig.savefig(
    FIG_DIR
    / "fig03_list_reliability.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## 29. Reliability figure — summary

In [ ]:
task = "summary"

subset = (
    task_oof_df[
        task_oof_df[
            "type"
        ] == task
    ]
)

fig, ax = plt.subplots(
    figsize=(6.4, 6.0)
)

ax.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Perfect calibration",
)

for method in [
    "task_ridge_raw",
    "task_ridge_isotonic",
]:
    cal = calibration_table(
        subset[
            "task_score"
        ],
        subset[
            method
        ],
    )

    ax.plot(
        cal[
            "mean_confidence"
        ],
        cal[
            "mean_observed_task_score"
        ],
        marker="o",
        label=method,
    )

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

ax.set_xlabel(
    "Predicted expected ROUGE-L"
)

ax.set_ylabel(
    "Observed mean ROUGE-L"
)

ax.set_title(
    "Summary task-specific reliability"
)

ax.legend()

fig.tight_layout()

fig.savefig(
    FIG_DIR
    / "fig04_summary_reliability.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# Part F — Within-task discrimination

Phase 4E requires confidence to rank questions **within the same task**.

We therefore inspect confidence deciles separately for each task.

## 30. Task-specific confidence deciles

In [ ]:
decile_rows = []

for task in TASKS:
    subset = (
        task_oof_df[
            task_oof_df[
                "type"
            ] == task
        ]
        .copy()
    )

    # Raw Ridge is used for ordering to avoid isotonic plateaus.
    subset[
        "quality_decile"
    ] = pd.qcut(
        subset[
            "task_ridge_raw"
        ],
        q=10,
        labels=False,
        duplicates="drop",
    )

    grouped = (
        subset.groupby(
            "quality_decile"
        )
        .agg(
            n=("id", "size"),
            mean_raw_score=(
                "task_ridge_raw",
                "mean"
            ),
            mean_calibrated_confidence=(
                "task_ridge_isotonic",
                "mean"
            ),
            mean_observed_task_score=(
                "task_score",
                "mean"
            ),
        )
        .reset_index()
    )

    grouped[
        "task"
    ] = task

    decile_rows.append(
        grouped
    )

task_deciles_df = pd.concat(
    decile_rows,
    ignore_index=True
)

task_deciles_df.to_csv(
    TABLE_DIR
    / "task_specific_confidence_deciles.csv",
    index=False
)

display(
    task_deciles_df
)

## 31. Within-task decile figure

In [ ]:
fig, ax = plt.subplots(
    figsize=(9.0, 5.5)
)

for task in TASKS:
    part = (
        task_deciles_df[
            task_deciles_df[
                "task"
            ] == task
        ]
    )

    ax.plot(
        part[
            "quality_decile"
        ],
        part[
            "mean_observed_task_score"
        ],
        marker="o",
        label=task,
    )

ax.set_xlabel(
    "Raw-Ridge quality decile"
)

ax.set_ylabel(
    "Observed mean task score"
)

ax.set_ylim(
    0,
    1
)

ax.set_title(
    "Within-task answer quality across confidence deciles"
)

ax.legend()

fig.tight_layout()

fig.savefig(
    FIG_DIR
    / "fig05_within_task_confidence_deciles.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# Part G — Paired bootstrap

We compare:

1. isotonic vs raw Ridge;
2. raw Ridge vs train-mean baseline;
3. raw Ridge vs grounding-only baselines.

For MAE and MSE, negative `model_a − model_b` is favorable.

## 32. Bootstrap helper

In [ ]:
def bootstrap_error_difference(
    y_true,
    pred_a,
    pred_b,
    n_boot=BOOTSTRAP_REPS,
    seed=SEED,
):
    y_true = np.asarray(
        y_true,
        dtype=float
    )

    pred_a = np.asarray(
        pred_a,
        dtype=float
    )

    pred_b = np.asarray(
        pred_b,
        dtype=float
    )

    n = len(y_true)

    rng = np.random.default_rng(
        seed
    )

    mae_delta = []
    mse_delta = []

    for _ in range(
        n_boot
    ):
        idx = rng.integers(
            0,
            n,
            size=n
        )

        err_a = (
            pred_a[
                idx
            ]
            - y_true[
                idx
            ]
        )

        err_b = (
            pred_b[
                idx
            ]
            - y_true[
                idx
            ]
        )

        mae_delta.append(
            np.mean(
                np.abs(
                    err_a
                )
            )
            - np.mean(
                np.abs(
                    err_b
                )
            )
        )

        mse_delta.append(
            np.mean(
                err_a ** 2
            )
            - np.mean(
                err_b ** 2
            )
        )

    mae_delta = np.asarray(
        mae_delta
    )

    mse_delta = np.asarray(
        mse_delta
    )

    return {
        "mae_mean_delta_a_minus_b": float(
            mae_delta.mean()
        ),
        "mae_ci95_low": float(
            np.quantile(
                mae_delta,
                0.025
            )
        ),
        "mae_ci95_high": float(
            np.quantile(
                mae_delta,
                0.975
            )
        ),
        "mse_mean_delta_a_minus_b": float(
            mse_delta.mean()
        ),
        "mse_ci95_low": float(
            np.quantile(
                mse_delta,
                0.025
            )
        ),
        "mse_ci95_high": float(
            np.quantile(
                mse_delta,
                0.975
            )
        ),
    }

## 33. Run task-specific bootstrap comparisons

In [ ]:
bootstrap_rows = []

comparisons = [
    (
        "task_ridge_isotonic",
        "task_ridge_raw",
    ),
    (
        "task_ridge_raw",
        "train_mean_baseline",
    ),
    (
        "task_ridge_raw",
        "mean_top1_entailment",
    ),
    (
        "task_ridge_raw",
        "top1_supported_claim_rate",
    ),
]

for task_index, task in enumerate(
    TASKS
):
    subset = (
        task_oof_df[
            task_oof_df[
                "type"
            ] == task
        ]
    )

    for comp_index, (
        model_a,
        model_b
    ) in enumerate(
        comparisons
    ):
        stats = (
            bootstrap_error_difference(
                subset[
                    "task_score"
                ],
                subset[
                    model_a
                ],
                subset[
                    model_b
                ],
                seed=(
                    SEED
                    + 100 * task_index
                    + comp_index
                ),
            )
        )

        stats.update({
            "task": task,
            "model_a": model_a,
            "model_b": model_b,
            "n": int(
                len(subset)
            ),
        })

        bootstrap_rows.append(
            stats
        )

task_bootstrap_df = pd.DataFrame(
    bootstrap_rows
)

task_bootstrap_df.to_csv(
    TABLE_DIR
    / "task_specific_paired_bootstrap_errors.csv",
    index=False
)

display(
    task_bootstrap_df
)

# Part H — Optional comparison with pooled Phase 4D v1

This section is descriptive only.

It demonstrates why the pooled model is not used for Phase 4E.

## 34. Load pooled v1 summary if available

In [ ]:
if POOLED_V1_SUMMARY.exists():
    pooled_v1_summary = pd.read_csv(
        POOLED_V1_SUMMARY
    )

    display(
        pooled_v1_summary
    )
else:
    pooled_v1_summary = None

    print(
        "Pooled Phase 4D v1 summary not found; "
        "comparison skipped."
    )

# Part I — Refit frozen task-specific models for future locked-test use

For each task:

1. fit Ridge on all available validation questions of that task;
2. fit isotonic calibration on that task's **cross-fitted raw OOF predictions**;
3. save both artifacts.

No locked-test label is used to update these models later.

## 35. Fit final task-specific Ridge and isotonic models

In [ ]:
final_model_manifest = {}

for task in TASKS:
    task_df = (
        model_df[
            model_df[
                "type"
            ] == task
        ]
        .copy()
    )

    task_oof = (
        task_oof_df[
            task_oof_df[
                "type"
            ] == task
        ]
        .copy()
    )

    X_task = (
        task_df[
            NUMERIC_FEATURES
        ]
    )

    y_task = (
        task_df[
            "task_score"
        ].to_numpy(float)
    )

    final_ridge = (
        make_ridge_pipeline()
    )

    final_ridge.fit(
        X_task,
        y_task,
    )

    ridge_path = (
        MODEL_DIR
        / f"{task}_ridge_confidence_model.joblib"
    )

    joblib.dump(
        final_ridge,
        ridge_path
    )

    # Important: calibrator is trained on OOF raw predictions.
    final_isotonic = (
        IsotonicRegression(
            y_min=0.0,
            y_max=1.0,
            out_of_bounds="clip",
        )
    )

    final_isotonic.fit(
        task_oof[
            "task_ridge_raw"
        ].to_numpy(float),
        task_oof[
            "task_score"
        ].to_numpy(float),
    )

    iso_path = (
        MODEL_DIR
        / f"{task}_isotonic_calibrator.joblib"
    )

    joblib.dump(
        final_isotonic,
        iso_path
    )

    final_model_manifest[
        task
    ] = {
        "n_validation": int(
            len(task_df)
        ),
        "ridge_model": str(
            ridge_path
        ),
        "isotonic_calibrator": str(
            iso_path
        ),
        "ranking_score": (
            "raw Ridge prediction"
        ),
        "reported_confidence": (
            "isotonic-calibrated expected task score"
        ),
    }

print(
    json.dumps(
        final_model_manifest,
        indent=2,
        ensure_ascii=False
    )
)

# Part J — Freeze Phase 4D v2

## 36. Freeze confidence method

In [ ]:
frozen_confidence_method = {
    "dataset": "BioASQ14b",
    "validation_questions": 862,
    "internal_test_evaluated": False,

    "frozen_evidence_strategy": (
        FROZEN_EVIDENCE_STRATEGY
    ),

    "design": (
        "task-specific confidence estimation"
    ),

    "reason_for_v2": (
        "Pooled expected-score confidence was "
        "confounded by heterogeneous task-score scales "
        "across yes/no, factoid, list, and summary."
    ),

    "question_type_used_as_feature": False,

    "tasks": {
        "yesno": {
            "n": int(
                (
                    model_df[
                        "type"
                    ] == "yesno"
                ).sum()
            ),
            "target": YESNO_COL,
            "interpretation": (
                "expected correctness"
            ),
        },
        "factoid": {
            "n": int(
                (
                    model_df[
                        "type"
                    ] == "factoid"
                ).sum()
            ),
            "target": FACTOID_COL,
            "interpretation": (
                "expected MRR"
            ),
        },
        "list": {
            "n": int(
                (
                    model_df[
                        "type"
                    ] == "list"
                ).sum()
            ),
            "target": LIST_COL,
            "interpretation": (
                "expected F1"
            ),
        },
        "summary": {
            "n": int(
                (
                    model_df[
                        "type"
                    ] == "summary"
                ).sum()
            ),
            "target": SUMMARY_COL,
            "interpretation": (
                "expected ROUGE-L"
            ),
        },
    },

    "features": NUMERIC_FEATURES,

    "base_model": {
        "type": "Ridge",
        "alpha": RIDGE_ALPHA,
    },

    "validation_protocol": {
        "outer_folds": OUTER_FOLDS,
        "inner_folds": INNER_FOLDS,
        "yesno_splitter": (
            "StratifiedKFold on binary target"
        ),
        "other_task_splitter": (
            "shuffled KFold"
        ),
        "isotonic_training": (
            "inner OOF predictions within each outer fold"
        ),
    },

    "frozen_operational_outputs": {
        "ranking_signal": (
            "task-specific raw Ridge score"
        ),
        "reported_confidence": (
            "task-specific isotonic-calibrated "
            "expected task score"
        ),
        "isotonic_plateau_tie_break": (
            "raw Ridge score"
        ),
    },

    "baselines": [
        "outer-training-fold mean target",
        "mean_top1_entailment",
        "top1_supported_claim_rate",
    ],

    "final_model_manifest": (
        final_model_manifest
    ),

    "abstention_threshold_selected": False,

    "next_phase": (
        "Phase 4E — task-specific selective "
        "answering / abstention"
    ),
}

with open(
    PHASE4D_DIR
    / "frozen_confidence_method_v2.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        frozen_confidence_method,
        f,
        indent=2,
        ensure_ascii=False
    )

print(
    json.dumps(
        frozen_confidence_method,
        indent=2,
        ensure_ascii=False
    )
)

## 37. Phase 4D v2 report

In [ ]:
phase4d_v2_report = {
    "dataset": "BioASQ14b",
    "validation_questions": 862,
    "internal_test_evaluated": False,

    "method": (
        "task-specific Ridge quality scoring "
        "+ nested cross-fitted isotonic calibration"
    ),

    "task_specific_metrics": (
        task_metric_df.to_dict(
            orient="records"
        )
    ),

    "macro_task_summary": (
        macro_summary.to_dict(
            orient="records"
        )
    ),

    "yesno_binary_calibration": (
        yesno_binary_df.to_dict(
            orient="records"
        )
    ),

    "paired_bootstrap": (
        task_bootstrap_df.to_dict(
            orient="records"
        )
    ),

    "abstention_threshold_selected": False,

    "interpretation_note": (
        "The calibrated output represents expected "
        "benchmark task performance. For factoid/list/"
        "summary it is not a probability of clinical truth."
    ),
}

with open(
    PHASE4D_DIR
    / "phase4D_v2_report.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        phase4d_v2_report,
        f,
        indent=2,
        ensure_ascii=False
    )

print(
    json.dumps(
        phase4d_v2_report,
        indent=2,
        ensure_ascii=False
    )
)

# STOP POINT 4D-v2

Before Phase 4E, review and send:

1. **21. Evaluate every method within each task**
2. **22. Ridge raw vs isotonic summary**
3. **23. Yes/no binary calibration**
4. **24. Macro-average task-specific metrics**
5. **26–29 reliability diagrams**
6. **30. Task-specific confidence deciles**
7. **31. Within-task decile figure**
8. **33. Task-specific bootstrap comparisons**
9. `frozen_confidence_method_v2.json`
10. `phase4D_v2_report.json`

---

## Criteria for proceeding to Phase 4E

We are looking for evidence that the confidence score provides meaningful **within-task ranking**, not merely good marginal calibration.

Important diagnostics:

- raw Ridge Spearman within each task;
- monotonic increase in observed task score across raw-Ridge deciles;
- improvement over the train-mean baseline;
- reliability of isotonic expected-score confidence.

A task may still have weak discrimination. If so, Phase 4E should report that explicitly rather than forcing a useful abstention policy.

No locked-test evaluation is allowed here.

## 38. Final safety check

In [ ]:
if not ALLOW_INTERNAL_TEST_EVALUATION:
    print(
        "INTERNAL TEST REMAINS LOCKED."
    )
else:
    raise RuntimeError(
        "Do not unlock the internal test during Phase 4D v2."
    )

print(
    "Phase 4D v2 output:",
    PHASE4D_DIR
)

print(
    "Question type used as feature:",
    False
)

print(
    "Operational ranking signal:",
    "task-specific raw Ridge"
)

print(
    "Reported confidence:",
    "task-specific isotonic expected task score"
)

print(
    "Abstention threshold selected:",
    False
)